# Check BDF Files

Pengecekan ketersediaan file .bdf untuk setiap subject berdasarkan IDs di cleaned_transcript_mapping.csv

In [11]:
import pandas as pd
import os
from collections import defaultdict

# Get the project root directory
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(notebook_dir, '../../../'))

# Define paths
cleaned_csv_path = os.path.join(project_root, 'dataset/cleaned_transcript_mapping.csv')
raw_data_path = os.path.join(project_root, 'dataset/raw')

print(f"Project Root: {project_root}")
print(f"Cleaned CSV Path: {cleaned_csv_path}")
print(f"Raw Data Path: {raw_data_path}")

# Load the cleaned data
df = pd.read_csv(cleaned_csv_path)
print(f"\nData loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

Project Root: /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer
Cleaned CSV Path: /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/cleaned_transcript_mapping.csv
Raw Data Path: /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/raw

Data loaded successfully!
Shape: (1050, 4)
Columns: ['id', 'subject', 'sentence', 'gender']

First 5 rows:
      id subject                                  sentence gender
0  1_DAM     DAM                          siapa wanita itu   male
1  2_DAM     DAM     anda benar benar kakek yang murah hati   male
2  3_DAM     DAM                        cobalah besok pagi   male
3  4_DAM     DAM   selamat sore nona ada yang bisa dibantu   male
4  5_DAM     DAM  untung saja ada orang baik yang menolong   male


## Get Subject Information from CSV

In [12]:
# Group by subject and get unique features
subject_info = df.groupby('subject').first()[['gender']].reset_index()
print(f"Total subjects: {len(subject_info)}")
print("\nSubject Information:")
print(subject_info.to_string(index=False))

# Create a dictionary for subject -> gender mapping
subject_gender_map = dict(zip(subject_info['subject'], subject_info['gender']))

# Group IDs by subject
subject_ids = defaultdict(list)
for idx, row in df.iterrows():
    subject_ids[row['subject']].append(row['id'])

print(f"\n\nSubjects and their ID counts:")
for subject in sorted(subject_ids.keys()):
    print(f"  {subject}: {len(subject_ids[subject])} IDs")

Total subjects: 12

Subject Information:
subject gender
    BEL female
    DAM   male
    ERI   male
    EVE female
    FAR female
    KEN female
    LIA female
    MAR   male
    NAU   male
    RAN   male
    RET female
    SUL   male


Subjects and their ID counts:
  BEL: 100 IDs
  DAM: 50 IDs
  ERI: 50 IDs
  EVE: 100 IDs
  FAR: 100 IDs
  KEN: 100 IDs
  LIA: 100 IDs
  MAR: 100 IDs
  NAU: 100 IDs
  RAN: 50 IDs
  RET: 100 IDs
  SUL: 100 IDs


## Check BDF Files Existence

In [13]:
def check_bdf_files_for_subject(subject, gender, subject_ids, raw_data_path):
    """
    Check if BDF files exist for a given subject
    """
    # Build the path to the subject's bdf+ folder
    bdf_folder = os.path.join(raw_data_path, gender, subject, 'bdf+')
    
    result = {
        'subject': subject,
        'gender': gender,
        'bdf_folder': bdf_folder,
        'folder_exists': os.path.isdir(bdf_folder),
        'total_ids': len(subject_ids[subject]),
        'bdf_files_found': 0,
        'missing_ids': [],
        'found_ids': []
    }
    
    # If folder doesn't exist, return early
    if not result['folder_exists']:
        result['missing_ids'] = subject_ids[subject]
        return result
    
    # List all files in the bdf+ folder
    try:
        bdf_files = [f for f in os.listdir(bdf_folder) if f.endswith('.bdf')]
        result['bdf_files_found'] = len(bdf_files)
        
        # Extract IDs from filenames (assuming format: ID_*.bdf)
        bdf_ids = set()
        for bdf_file in bdf_files:
            # Extract the ID part (before the first underscore after the number)
            prefix = bdf_file.split('_')[0] + '_' + bdf_file.split('_')[1] if '_' in bdf_file else bdf_file.split('.')[0]
            bdf_ids.add(prefix)
        
        # Check for missing IDs
        for id_val in subject_ids[subject]:
            if any(bdf_file.startswith(id_val) and bdf_file.endswith('.bdf') for bdf_file in bdf_files):
                result['found_ids'].append(id_val)
            else:
                result['missing_ids'].append(id_val)
                
    except Exception as e:
        result['error'] = str(e)
        result['missing_ids'] = subject_ids[subject]
    
    return result

# Check all subjects
check_results = []
for subject in sorted(subject_ids.keys()):
    gender = subject_gender_map[subject]
    result = check_bdf_files_for_subject(subject, gender, subject_ids, raw_data_path)
    check_results.append(result)

print(f"BDF file checking completed for {len(check_results)} subjects")

BDF file checking completed for 12 subjects


## Report Results

In [14]:
print("\n" + "=" * 100)
print("BDF FILE CHECK REPORT")
print("=" * 100)

total_subjects = len(check_results)
subjects_with_complete_files = 0
subjects_with_missing_files = 0
subjects_with_missing_folder = 0
total_missing_files = 0

for result in check_results:
    print(f"\n{'-' * 100}")
    print(f"SUBJECT: {result['subject']} ({result['gender']})")
    print(f"{'-' * 100}")
    
    print(f"BDF Folder: {result['bdf_folder']}")
    print(f"Folder Exists: {result['folder_exists']}")
    
    if not result['folder_exists']:
        print(f"⚠️  FOLDER NOT FOUND!")
        print(f"   Missing all {result['total_ids']} IDs")
        subjects_with_missing_folder += 1
        total_missing_files += result['total_ids']
    else:
        print(f"Total IDs expected: {result['total_ids']}")
        print(f"BDF files found: {result['bdf_files_found']}")
        print(f"IDs with matching BDF files: {len(result['found_ids'])}")
        print(f"IDs missing BDF files: {len(result['missing_ids'])}")
        
        if len(result['missing_ids']) == 0:
            print(f"✓ All BDF files present!")
            subjects_with_complete_files += 1
        else:
            print(f"✗ Missing BDF files for:")
            for missing_id in sorted(result['missing_ids'])[:10]:  # Show first 10
                print(f"   - {missing_id}")
            if len(result['missing_ids']) > 10:
                print(f"   ... and {len(result['missing_ids']) - 10} more")
            subjects_with_missing_files += 1
            total_missing_files += len(result['missing_ids'])

print(f"\n\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(f"Total subjects: {total_subjects}")
print(f"Subjects with all BDF files: {subjects_with_complete_files}")
print(f"Subjects with missing BDF files: {subjects_with_missing_files}")
print(f"Subjects with missing folder: {subjects_with_missing_folder}")
print(f"Total missing BDF files: {total_missing_files}")
print("=" * 100)


BDF FILE CHECK REPORT

----------------------------------------------------------------------------------------------------
SUBJECT: BEL (female)
----------------------------------------------------------------------------------------------------
BDF Folder: /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/raw/female/BEL/bdf+
Folder Exists: True
Total IDs expected: 100
BDF files found: 100
IDs with matching BDF files: 100
IDs missing BDF files: 0
✓ All BDF files present!

----------------------------------------------------------------------------------------------------
SUBJECT: DAM (male)
----------------------------------------------------------------------------------------------------
BDF Folder: /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/raw/male/DAM/bdf+
Folder Exists: True
Total IDs expected: 50
BDF files found: 51
IDs with matching BDF files: 50
IDs missing BDF files: 0
✓ All BDF files present!

---

## Detailed Missing Files Report

In [15]:
# Create a detailed report of missing files
missing_report = []

for result in check_results:
    if result['folder_exists'] and len(result['missing_ids']) > 0:
        for missing_id in result['missing_ids']:
            missing_report.append({
                'subject': result['subject'],
                'gender': result['gender'],
                'missing_id': missing_id,
                'bdf_folder': result['bdf_folder']
            })

if missing_report:
    missing_df = pd.DataFrame(missing_report)
    print(f"\nDetailed Missing Files ({len(missing_df)} total):")
    print(missing_df.to_string(index=False))
else:
    print("\nNo missing files found!")


Detailed Missing Files (4 total):
subject gender missing_id                                                                                             bdf_folder
    SUL   male     43_SUL /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/raw/male/SUL/bdf+
    SUL   male     44_SUL /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/raw/male/SUL/bdf+
    SUL   male     82_SUL /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/raw/male/SUL/bdf+
    SUL   male     85_SUL /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/raw/male/SUL/bdf+


## Investigate Subjects with Missing Data - Check for Duplicate Prefixes

In [16]:
print("\n" + "=" * 100)
print("INVESTIGATION: SUBJECTS WITH MISSING DATA - CHECKING FOR DUPLICATE PREFIXES")
print("=" * 100)

# Filter results for subjects with missing files
subjects_with_issues = [r for r in check_results if r['folder_exists'] and len(r['missing_ids']) > 0]

if not subjects_with_issues:
    print("\nNo subjects with missing files found.")
else:
    for result in subjects_with_issues:
        subject = result['subject']
        gender = result['gender']
        bdf_folder = result['bdf_folder']
        missing_ids = result['missing_ids']
        
        print(f"\n{'-' * 100}")
        print(f"SUBJECT: {subject} ({gender})")
        print(f"Folder: {bdf_folder}")
        print(f"Missing IDs: {len(missing_ids)}")
        print(f"{'-' * 100}")
        
        # List all BDF files in the folder
        try:
            all_bdf_files = sorted([f for f in os.listdir(bdf_folder) if f.endswith('.bdf')])
            print(f"\nTotal BDF files found: {len(all_bdf_files)}")
            
            # Extract prefixes from all BDF files
            prefix_to_files = defaultdict(list)
            for bdf_file in all_bdf_files:
                # Extract prefix (ID part: e.g., "1_SUB", "2_SUB")
                parts = bdf_file.split('_')
                if len(parts) >= 2:
                    prefix = parts[0] + '_' + parts[1]  # Format: "1_SUB"
                    prefix_to_files[prefix].append(bdf_file)
            
            # Check for duplicate prefixes
            duplicate_prefixes = {p: files for p, files in prefix_to_files.items() if len(files) > 1}
            
            print(f"\nUnique prefixes: {len(prefix_to_files)}")
            print(f"Prefixes with duplicates: {len(duplicate_prefixes)}")
            
            if duplicate_prefixes:
                print(f"\n⚠️  DUPLICATE PREFIXES FOUND:")
                print("-" * 100)
                for prefix in sorted(duplicate_prefixes.keys()):
                    files = duplicate_prefixes[prefix]
                    print(f"\nPrefix: {prefix}")
                    print(f"Files ({len(files)}):")
                    for f in sorted(files):
                        print(f"  - {f}")
            else:
                print(f"\n✓ No duplicate prefixes found.")
            
            # Show which expected IDs are missing
            print(f"\n\nMissing IDs Analysis:")
            print("-" * 100)
            print(f"Expected IDs but not found: {len(missing_ids)}")
            for missing_id in sorted(missing_ids)[:20]:
                # Check if there are any files with similar prefix
                similar = [p for p in prefix_to_files.keys() if missing_id.split('_')[1] in p]
                status = f"Similar prefix exists: {similar}" if similar else "No similar files"
                print(f"  {missing_id}: {status}")
            if len(missing_ids) > 20:
                print(f"  ... and {len(missing_ids) - 20} more")
                
        except Exception as e:
            print(f"\nError investigating subject {subject}: {str(e)}")

print(f"\n\n" + "=" * 100)
print("INVESTIGATION COMPLETE")
print("=" * 100)


INVESTIGATION: SUBJECTS WITH MISSING DATA - CHECKING FOR DUPLICATE PREFIXES

----------------------------------------------------------------------------------------------------
SUBJECT: SUL (male)
Folder: /Users/steven/Documents/Github Repositories/EEG-to-Text_Conformer-Transducer/dataset/raw/male/SUL/bdf+
Missing IDs: 4
----------------------------------------------------------------------------------------------------

Total BDF files found: 100

Unique prefixes: 96
Prefixes with duplicates: 4

⚠️  DUPLICATE PREFIXES FOUND:
----------------------------------------------------------------------------------------------------

Prefix: 32_SUL
Files (2):
  - 32_SUL_EPOCX_731200_2026.03.04T12.04.20+07.00.md.bdf
  - 32_SUL_EPOCX_731200_2026.03.04T12.45.41+07.00.md.bdf

Prefix: 35_SUL
Files (2):
  - 35_SUL_EPOCX_731200_2026.03.04T12.05.14+07.00.md.bdf
  - 35_SUL_EPOCX_731200_2026.03.04T12.46.36+07.00.md.bdf

Prefix: 53_SUL
Files (2):
  - 53_SUL_EPOCX_731200_2026.03.04T12.10.36+07.00.md.bdf